# Chapter 12: Stream Processing
*From: Designing Data-Intensive Applications*

This notebook is a study companion for Chapter 12. It treats a stream-processing platform as a system to be designed: messaging substrate, ingestion, derived-state pipelines, time and joins, and the fault-tolerance machinery that makes any of it usable in production.

---

## TL;DR

- A **stream** is unbounded data that arrives incrementally; the streaming counterpart of a file in a batch job is an **event log**.
- Two broker families dominate: **AMQP/JMS-style** brokers (per-message ack, destructive read, message-level load balancing) and **log-based** brokers like **Kafka** (append-only partitioned log, replayable, shard-level assignment).
- Once writes are captured as a log — via **change data capture (CDC)** or **event sourcing** — derived systems (search index, cache, warehouse, materialized views) can stay in sync by being followers of that log.
- Time on streams is hard: **event time vs processing time**, **straggler events**, and **window types** (tumbling, hopping, sliding, session) all show up.
- Joins come in three shapes — **stream–stream** (window join), **stream–table** (enrichment via local CDC-fed lookup), **table–table** (materialized view maintenance).
- **Exactly-once** is achieved as *effectively-once*: microbatching/checkpoints + atomic commit *inside* the framework, or **idempotent writes** stamped with monotonic offsets.

---

## Functional Requirements

- Producers can publish events to named topics; multiple producers per topic.
- Consumers can subscribe in two modes:
  - **Load balancing** — split work across a consumer group.
  - **Fan-out** — every consumer group sees every event.
- Replay: a new consumer can start from an arbitrary offset (including offset 0) and re-read history without disturbing other consumers.
- Ordering guarantees within a partition; routing by partition key (e.g., user-id) preserves per-entity order.
- Support both **transient messaging** (drop after ack, AMQP-style) and **durable log** (retain for days/weeks, replay on demand).
- Stream-processing operators that maintain state: aggregations, windowed joins, materialized views.

## Non-Functional Requirements

- Throughput: millions of events/sec across the cluster (Kafka benchmarks routinely cite ~2M writes/sec on three machines).
- End-to-end latency: sub-second typical, single-digit ms achievable with care.
- Durability: fsync + replication; broker survives single-node loss.
- Backpressure: behavior under slow consumers must be defined (drop, buffer to disk, or block producer).
- Fault tolerance: operator failures must not lose state or double-apply external side effects.

---

## Back-of-Envelope: Kafka Disk Retention

The chapter's example: a 20 TB drive with 250 MB/s sequential write throughput. How long can a single broker disk buffer messages before old segments must be evicted?

In [ ]:
# Single-disk Kafka buffer capacity
disk_capacity_tb = 20
disk_capacity_bytes = disk_capacity_tb * 1024**4
write_throughput_mbps = 250
write_throughput_bps = write_throughput_mbps * 1024**2

seconds_to_fill = disk_capacity_bytes / write_throughput_bps
hours_to_fill = seconds_to_fill / 3600
print(f"Time to fill 20 TB at 250 MB/s: {hours_to_fill:.1f} h ({seconds_to_fill/86400:.2f} days)")

# In practice, brokers run well below peak write rate.
for utilization in (1.0, 0.5, 0.25, 0.10):
    effective_bps = write_throughput_bps * utilization
    days = disk_capacity_bytes / effective_bps / 86400
    print(f"  at {int(utilization*100):3d}% of peak write: {days:6.1f} days of retention")

Takeaway: even at full write bandwidth, a single disk gives ~22 hours of buffer. Real deployments rarely sustain peak writes, so days-to-weeks of replay is normal — enough time for a human operator to notice and rescue a slow consumer before it falls off the back of the log.

## Back-of-Envelope: Cluster-Level Throughput

Sharding the log across N brokers multiplies both storage and write bandwidth roughly linearly.

In [ ]:
# Cluster sizing for a target ingest rate
brokers = 12
disks_per_broker = 8
disk_capacity_tb = 20
peak_write_mbps_per_disk = 250
replication_factor = 3
target_avg_event_bytes = 500

raw_capacity_tb = brokers * disks_per_broker * disk_capacity_tb
usable_capacity_tb = raw_capacity_tb / replication_factor
peak_cluster_write_gbps = (
    brokers * disks_per_broker * peak_write_mbps_per_disk / 1024
)
# Each event written 'replication_factor' times — usable producer rate is divided.
producer_throughput_gbps = peak_cluster_write_gbps / replication_factor
producer_events_per_sec = producer_throughput_gbps * 1024**3 / target_avg_event_bytes

print(f"Raw capacity:        {raw_capacity_tb:,} TB")
print(f"Usable (RF={replication_factor}):    {usable_capacity_tb:,.0f} TB")
print(f"Peak disk write:     {peak_cluster_write_gbps:,.1f} GB/s")
print(f"Producer throughput: {producer_throughput_gbps:,.1f} GB/s")
print(f"Events/sec @ {target_avg_event_bytes} B:  {producer_events_per_sec:,.0f}")

# Retention at a given sustained ingest
sustained_events_per_sec = 1_000_000
sustained_bps = sustained_events_per_sec * target_avg_event_bytes
retention_days = (usable_capacity_tb * 1024**4) / sustained_bps / 86400
print(f"Retention @ 1M events/s: {retention_days:.1f} days")

---

## High-Level Architecture

```mermaid
graph LR
  subgraph Producers
    App[App Servers]
    DB[(OLTP DB)]
    Sensors[Sensors / Mobile]
  end

  subgraph Broker[Log-Based Broker - Kafka-style]
    P0[Partition 0]
    P1[Partition 1]
    P2[Partition 2]
    P3[Partition 3]
  end

  subgraph DerivedSystems[Derived Systems]
    Search[Search Index]
    Cache[Cache]
    DW[(Warehouse)]
    MV[Materialized Views]
  end

  App -->|publish| Broker
  DB -->|CDC connector| Broker
  Sensors -->|publish| Broker

  Broker -->|consumer group A| Search
  Broker -->|consumer group B| Cache
  Broker -->|consumer group C| DW
  Broker -->|stream operators| MV
```

The broker is the spine. Once writes — including database writes via CDC — flow through it, every other store becomes a *follower* of a single ordered log instead of trying to keep itself in sync with peers (the dual-write trap, see below).

---

## 1. Message Brokers: AMQP/JMS vs Log-Based

The fundamental split: does receiving a message *delete* it, or does it merely advance an offset?

| Dimension | AMQP/JMS (RabbitMQ, ActiveMQ, IBM MQ) | Log-Based (Kafka, Kinesis, Pulsar) |
|-----------|----------------------------------------|-------------------------------------|
| Storage model | Queue, transient; deleted on ack | Append-only log on disk, retained |
| Delivery unit | Per-message assignment to consumers | Per-partition assignment to consumer node |
| Load balancing granularity | Message-level (broker picks consumer) | Shard-level (whole partition → one node) |
| Ordering | Best-effort; broken by redelivery | Total order *within* a partition |
| Replay | No — read is destructive | Yes — rewind offset, no impact on others |
| Slow message handling | OK — other consumers grab next message | Head-of-line blocking within partition |
| Backpressure / overflow | DLQ, drop, or unbounded queue growth | Bounded ring buffer (disk size) |
| Fan-out | Multiple subscriptions / exchanges | Each consumer group has independent offsets |
| Best fit | Task queues, expensive per-message work, ordering not critical | High-throughput pipelines, derived data, replay needed |

### Log-based broker: producer/consumer flow

```mermaid
sequenceDiagram
  participant P as Producer
  participant B as Broker (partition k)
  participant CG1 as Consumer Group A (offset 1042)
  participant CG2 as Consumer Group B (offset 87)

  P->>B: append(event) -> offset=1100
  B-->>P: ack
  CG1->>B: fetch(from=1042)
  B-->>CG1: events[1042..1100]
  CG1->>B: commit_offset(1100)
  CG2->>B: fetch(from=87)
  B-->>CG2: events[87..200]
  Note over CG1,CG2: independent offsets, no contention
```

### The trade-off

- **Per-message assignment** (AMQP) is great when each message is expensive and order doesn't matter — slow consumers get fewer messages, fast ones get more.
- **Per-shard assignment** (log) is great when messages are cheap, ordering matters, and replay matters — but a single slow message head-of-lines its whole partition.

Modern systems (Pulsar, Kafka KIP-932) blur the line — log-backed storage with queue-style per-message acknowledgment — but the mental model still helps when picking defaults.

See `[[01-message-brokers-amqp-vs-log]]`.

---

## 2. Change Data Capture (CDC)

Problem: applications need data in many shapes — OLTP row store, search index, cache, warehouse. The naive solution, **dual writes**, is broken.

### Why dual writes fail

```mermaid
sequenceDiagram
  participant C1 as Client 1 (X=A)
  participant C2 as Client 2 (X=B)
  participant DB as Database
  participant SI as Search Index

  C1->>DB: write X=A
  C2->>DB: write X=B
  Note right of DB: DB final: X=B
  C2->>SI: write X=B
  C1->>SI: write X=A
  Note right of SI: Index final: X=A
  Note over DB,SI: Permanently inconsistent, no error raised
```

The two systems disagree because the writes are interleaved differently at each. Plus partial-failure: one side may simply fail to apply.

### CDC turns the DB into the single leader

```mermaid
graph LR
  App[App] -->|writes| DB[(OLTP DB)]
  DB -->|replication log| CDC[CDC Connector<br/>e.g. Debezium]
  CDC -->|change events| Log[Log Broker]
  Log --> Search[Search Index]
  Log --> Cache[Cache]
  Log --> DW[(Data Warehouse)]
  Log --> ML[Feature Store]
```

The DB decides write order. Every consumer sees that *same* order. Inconsistency becomes impossible by construction; lag is the only failure mode.

| Approach | Race condition risk | Schema coupling | Operational simplicity | Replay |
|----------|---------------------|-----------------|------------------------|--------|
| Dual writes | High (interleaving) | None — app controls each write | Looks easy, breaks under load | None |
| CDC (log-tail) | None — DB log is single source | Internal DB schema becomes a public API | Needs connector, snapshot, log retention | Yes — replay log compaction or snapshot + tail |
| Outbox pattern | None — app writes outbox row in same txn | Outbox schema is the contract | Extra table, extra writes | Yes (outbox is logged) |

### Bootstrapping a new consumer

Two options:

1. **Initial snapshot + tail** — copy DB at offset T₀, then apply log from T₀ forward. (Debezium uses Netflix's DBLog incremental snapshot.)
2. **Log compaction** — broker keeps only the most recent value for each key, so replaying from offset 0 reconstructs current state without a snapshot.

See `[[02-change-data-capture]]`.

---

## 3. Event Sourcing & Immutable Event Logs

Same machinery as CDC, but applied at a higher level: the application *itself* writes intent-bearing immutable events as its source of truth, not low-level row diffs.

```mermaid
graph TD
  User[User Action] -->|append| EventLog[(Immutable Event Log<br/>system of record)]
  EventLog --> View1[Read View 1<br/>e.g. user profile DB]
  EventLog --> View2[Read View 2<br/>e.g. recommendations]
  EventLog --> View3[Read View 3<br/>e.g. analytics cube]
  EventLog --> View4[New View<br/>built later from scratch]
```

State and stream are duals: state is the *integral* of the event stream over time; the stream is the *derivative* of state.

### Why this is powerful

- **Auditability** — the log contains everything that ever happened, including cancelled actions (cart-add then cart-remove tells you about *consideration*, not just outcomes).
- **Schema evolution** — to introduce a new feature, derive a new read view from the same log; run old and new side by side; flip when ready.
- **Concurrency control** — the log linearizes writes, so a single-threaded shard consumer needs no locks (it sees one event at a time).
- **Accountant's ledger analogy** — never erase a wrong entry; append a compensating entry. The wrong entry stays for forensics.

### Trade-offs

| Pro | Con |
|-----|-----|
| Single source of truth | Read-your-writes lag for derived views (CQRS asynchronous gap) |
| Derive new views cheaply | Storage grows with churn; compaction may not apply (intent-level events) |
| Strong audit trail | GDPR / right-to-erasure conflicts with append-only — need crypto-shredding |
| Simplifies multi-object writes | Existing apps require a major rewrite |

### Crypto-shredding sketch

You can't truly delete an immutable event without rewriting history. Workaround:

```mermaid
graph LR
  PII[PII Event] -->|encrypt with K_user| Log[(Event Log)]
  KS[Key Store] -.->|holds K_user| Reader
  Log --> Reader
  Reader -->|decrypt| Plain[Plain Event]
  Delete[GDPR Delete] -.->|forget K_user| KS
  KS -.->|key gone| Reader2[Future Reader]
  Reader2 --> Garbage[Unreadable bytes]
```

The data is still on disk; the *key* is the mutable handle. Coarse-grained: you crypto-shred *all* data under one key or none of it.

See `[[03-event-sourcing-immutable-logs]]`.

---

## 4. Uses of Stream Processing

Four major patterns, distinguished by what kind of question is being asked of the stream:

| Pattern | Query/Data Model | State Required | Window Need | Example Systems |
|---------|------------------|----------------|-------------|-----------------|
| **Complex event processing (CEP)** | Long-lived queries; events flow through them and trigger matches | Pattern state machines | Often per-pattern bounded | Esper, Apama, Flink SQL |
| **Stream analytics** | Aggregations / statistics over windows | Counters, sketches (HyperLogLog, percentile) | Tumbling/hopping/sliding/session | Storm, Flink, Spark Streaming, Beam, Kafka Streams |
| **Materialized view maintenance** | Equivalent of a SQL view, kept fresh incrementally | Full state of the view | Effectively unbounded (back to t=0) | Kafka Streams, ksqlDB, Materialize, RisingWave, ClickHouse, Feldera |
| **Search on streams** | Standing search query; stream of documents | Indexed queries | Per-document, no time window | Elasticsearch percolator, Luwak |

### The inverted query model

```mermaid
graph LR
  subgraph Database[Traditional DB]
    DataDB[(Data, persisted)]
    Q1[Query, transient] --> DataDB --> Results1[Result, transient]
  end

  subgraph Stream[Stream Processor]
    Q2[Queries, persisted]
    Events[Events, transient] --> Q2 --> Results2[Match, transient]
  end
```

Traditional DBs persist data and let queries come and go. Stream processors persist *queries* and let events flow through. This is why CEP, stream analytics, and stream search all feel structurally similar — they're all instances of "data flows into a long-lived query."

### Incremental View Maintenance (IVM)

Plain `REFRESH MATERIALIZED VIEW` reprocesses the whole base table — too slow and too stale for streams. IVM systems compile SQL into operators that update only the rows changed by each event. Recent updates buffer in memory; reads merge buffered updates with on-disk view state. This is what blurs the line between "stream processor" and "database" in tools like Materialize and RisingWave.

See `[[04-stream-processing-use-cases]]`.

---

## 5. Reasoning About Time & Windows

### Two clocks

- **Event time** — when the event actually occurred (timestamp inside the event).
- **Processing time** — wall clock on the box doing the processing.

Using processing time is simple but produces lies whenever there's lag — restart a consumer, watch the backlog drain, and your "requests per second" graph shows a giant fake spike.

### The straggler problem

Events for the 10:37 minute keep arriving until... when? You have two strategies:

```mermaid
graph TD
  Event[Event with t=10:37:42] -->|arrives at processing 10:39:15| Decision{10:37 window<br/>already closed?}
  Decision -->|No| Add[Add to window]
  Decision -->|Yes, drop strategy| Drop[Drop, increment dropped-events metric]
  Decision -->|Yes, correction strategy| Retract[Retract previous output<br/>publish updated value]
```

A common trick: producers periodically emit a watermark — *"no more events with timestamp earlier than t will arrive from me"*. Consumers track watermarks per producer and close windows when *all* producers have advanced past `t`.

### Untrusted device clocks: the three-timestamp pattern

Mobile clocks lie. Use:

1. `t_event_device` — when the user did the thing (device clock)
2. `t_send_device` — when the device sent the batch (device clock)
3. `t_receive_server` — when the server got the batch (trusted server clock)

Estimated server-clock event time:

In [ ]:
# Adjusting an untrusted device clock against a trusted server clock
def correct_event_time(t_event_device, t_send_device, t_receive_server):
    """Return server-clock estimate of when the event actually occurred."""
    device_offset = t_send_device - t_receive_server  # how far ahead device clock runs
    return t_event_device - device_offset

# Example: device clock is 5 minutes fast
t_event_device   = 10_000  # seconds since epoch (device says event at 10000)
t_send_device    = 10_300  # device says it sent at 10300
t_receive_server = 10_000  # server actually received at 10000 -> device is 300s ahead
print(correct_event_time(t_event_device, t_send_device, t_receive_server))
# -> 9700  (server-clock time of the event, accounting for the 300s skew)

Assumptions: network delay is negligible compared to required accuracy, and the device clock skew didn't change between the event and the upload.

### Window types

| Window | Length | Overlap | State per key | Typical use |
|--------|--------|---------|---------------|-------------|
| **Tumbling** | Fixed | None — disjoint | One window's worth | Per-minute counts, billing buckets |
| **Hopping** | Fixed | Yes — fixed hop size | `length / hop` overlapping windows | Smoothed rolling averages |
| **Sliding** | Fixed | Continuous — any pair within length | Buffer of all in-window events | "events within 5 min of each other" |
| **Session** | Variable — closes on inactivity gap | None | Per active session | User sessions, conversion funnels |

Visual:

```mermaid
gantt
  title Window types over the same event stream
  dateFormat  HH:mm
  axisFormat  %H:%M
  section Tumbling 5m
  W1 :a1, 10:00, 5m
  W2 :a2, 10:05, 5m
  W3 :a3, 10:10, 5m
  section Hopping 5m / hop 1m
  H1 :b1, 10:00, 5m
  H2 :b2, 10:01, 5m
  H3 :b3, 10:02, 5m
  H4 :b4, 10:03, 5m
  section Session (inactivity 2m)
  S1 :c1, 10:00, 3m
  S2 :c2, 10:06, 4m
```

See `[[05-reasoning-about-time-and-windows]]`.

---

## 6. Stream Joins

Three shapes, distinguished by what kind of input each side is:

| Join | Left input | Right input | State maintained | Time semantics |
|------|------------|-------------|------------------|----------------|
| **Stream–stream** (window join) | Activity stream | Activity stream | Both sides indexed by join key, bounded by window | Both windowed; emit on match or window expiry |
| **Stream–table** (enrichment) | Activity stream | CDC changelog of a table | Local hash table of latest value per key | Table window is "all of history"; stream window may be 0 |
| **Table–table** (materialized view) | CDC changelog | CDC changelog | Both sides as latest-value tables | Both unbounded; output is changelog of join |

### Stream–stream: search and click

```mermaid
graph LR
  Search[Search Events<br/>session_id, query] --> Joiner
  Click[Click Events<br/>session_id, result_url] --> Joiner
  Joiner[Window Join Operator<br/>window=1h, key=session_id]
  Joiner -->|matched within 1h| Matched[Click-through events]
  Joiner -->|window expired without match| Unclicked[Abandoned-search events]
```

Both unmatched and matched outcomes are useful: the click-through *rate* needs both the numerator and the denominator.

### Stream–table: activity enrichment via local CDC-fed lookup

```mermaid
graph LR
  Activity[Activity Stream<br/>user_id, action] -->|lookup user_id| Joiner
  Profiles[(User Profile DB)] -->|CDC changelog| LocalHT[Local Hash Table<br/>in stream processor]
  LocalHT --> Joiner
  Joiner --> Enriched[Enriched Activity<br/>user_id, action, name, country, ...]
```

A naive "call the DB on every event" implementation overloads the DB. Instead, materialize a local copy from the CDC changelog and join in-process.

### State sizing for windowed joins

In [ ]:
# How big is the per-operator state for a stream-stream join?
events_per_sec_left  = 1_000_000     # 1M search events/s
events_per_sec_right = 1_000_000     # 1M click events/s
window_seconds       = 3600          # 1-hour join window
bytes_per_event      = 200           # session_id + payload + index overhead
replication_factor   = 2             # for state replication

events_in_window = (events_per_sec_left + events_per_sec_right) * window_seconds
state_bytes = events_in_window * bytes_per_event * replication_factor
state_gb = state_bytes / 1024**3
state_tb = state_gb / 1024
print(f"In-window events:  {events_in_window:,}")
print(f"Replicated state:  {state_gb:,.1f} GB ({state_tb:,.2f} TB)")

# Distribute across operator instances
shards = 64
per_shard_gb = state_gb / shards
print(f"Per-shard state ({shards} shards): {per_shard_gb:,.1f} GB")

This is why window length matters so much: doubling the window doubles in-flight join state.

### Time dependence and slowly changing dimensions

When a profile changes (or a tax rate changes), which version do you join with? Two options:

| Option | Pro | Con |
|--------|-----|-----|
| Always use *current* state (live join) | Simple; minimal storage | Reprocessing yields different output — non-deterministic |
| Version-stamp every change; event references the version it saw | Deterministic; correct historical reprocessing | Log compaction must be disabled (need full history); larger state |
| Denormalize at write time (embed tax_rate in sale event) | Deterministic without history | Duplication; harder to fix retroactive errors |

See `[[06-stream-joins]]`.

---

## 7. Stream Fault Tolerance & Exactly-Once Semantics

A stream job runs forever, so "rerun from the start on failure" is not on the table. We need finer-grained recovery that produces output *as if* every event was processed exactly once — even though under the hood, retries happen.

### Four mechanisms

| Mechanism | How it works | Latency | Output trust |
|-----------|--------------|---------|--------------|
| **Microbatching** (Spark Streaming) | Cut the stream into ~1s mini-batches, treat each as a tiny batch job | ~batch_size (typically 1s) | Exactly-once *within* the framework; downstream side effects can double-fire |
| **Checkpoint barriers** (Flink) | Inject barriers in the stream; on barrier, snapshot operator state to durable storage | ~ms — barriers don't force a window | Exactly-once within framework; downstream needs idempotence/transactions |
| **Internal atomic commit** (Kafka transactions, Dataflow) | State updates + offset commits + downstream message produces all in one atomic txn | Higher (commit overhead) | Truly exactly-once *if everything is inside the framework* |
| **Idempotent writes** | Tag each external write with the source offset; receiver dedupes | Lowest | Exactly-once visible effect, but only if processing is deterministic and writes carry monotonic IDs |

### The microbatch latency–throughput trade-off

In [ ]:
# Microbatching: smaller batches -> lower latency, more scheduling overhead.
overhead_per_batch_ms = 5      # fixed scheduling/commit cost per microbatch
events_per_second     = 1_000_000

for batch_ms in (10, 100, 250, 500, 1000, 5000):
    batches_per_second = 1000 / batch_ms
    overhead_fraction  = (overhead_per_batch_ms * batches_per_second) / 1000
    useful_throughput  = events_per_second * (1 - overhead_fraction)
    end_to_end_latency = batch_ms + overhead_per_batch_ms  # roughly
    print(
        f"batch={batch_ms:>5} ms | "
        f"overhead={overhead_fraction*100:5.1f}% | "
        f"useful_eps={useful_throughput:>11,.0f} | "
        f"~latency={end_to_end_latency:>5} ms"
    )

Smaller batch = better latency, worse throughput; larger batch = better throughput, worse latency. Pick based on whether the consumer of the output is human (>=100ms is fine) or automated (sometimes <10ms required).

### Idempotence via offset-stamped writes

```mermaid
sequenceDiagram
  participant K as Kafka (offset 1042)
  participant Op as Stream Operator
  participant DB as External KV Store

  K->>Op: event @ offset=1042
  Op->>DB: PUT user_42 = {value, last_offset=1042}
  Note over DB: stored
  Note over Op: crash before offset commit
  Op->>K: restart, resume from offset 1041
  K->>Op: event @ offset=1042 (replayed)
  Op->>DB: PUT user_42 = {value, last_offset=1042}<br/>conditional: only if stored offset < 1042
  DB-->>Op: noop, already at 1042
```

Preconditions:
- **Deterministic** processing (same input → same output every time).
- **Replayable input** in the same order (log-based broker delivers this).
- **Fencing** against zombie operators — old "thought-dead but actually alive" operator must not race the new one writing to the same key.

### Local vs remote state for recovery

| Strategy | Recovery time | Latency at steady state | Coupling |
|----------|---------------|--------------------------|----------|
| Remote state (e.g. external KV) | Fast (no rebuild) | High (network round-trip per event) | Coupled to KV availability |
| Local state, periodic snapshot to durable storage (Flink) | Medium — fetch latest snapshot then replay tail | Low | Loose |
| Local state, replicated via changelog topic (Kafka Streams) | Medium — restore from changelog topic | Low | Coupled to broker |
| No state stored — rebuild from input replay (small windows) | Slow if window is long | Low | Loose |

### Consumer lag estimation

In [ ]:
# How far behind the head of the log is a consumer?
producer_rate_eps = 500_000           # producer events/sec into a partition
consumer_rate_eps = 450_000           # consumer drains slightly slower
event_size_bytes  = 500
outage_seconds    = 600               # consumer was down 10 minutes

# Backlog accumulated during the outage
backlog_events = producer_rate_eps * outage_seconds
backlog_bytes  = backlog_events * event_size_bytes
print(f"Backlog after {outage_seconds}s outage: {backlog_events:,} events ({backlog_bytes/1024**3:.1f} GB)")

# How long to catch up at the consumer's normal slack
slack_eps = consumer_rate_eps - producer_rate_eps  # negative -> can never catch up
if slack_eps <= 0:
    catchup_eps = consumer_rate_eps * 1.5          # assume burst capacity
    slack_eps = catchup_eps - producer_rate_eps
    print(f"Need burst mode: {catchup_eps:,} eps")
catchup_seconds = backlog_events / slack_eps
print(f"Catch-up time: {catchup_seconds/60:.1f} min")

# Will the consumer fall off the back of the log?
log_retention_hours = 22       # from our earlier disk math
retention_seconds   = log_retention_hours * 3600
if catchup_seconds > retention_seconds:
    print("DANGER: consumer will lose data — increase retention or scale out")
else:
    print(f"Safe — catch-up < retention ({catchup_seconds:.0f}s < {retention_seconds}s)")

See `[[07-stream-fault-tolerance]]`.

---

## Trade-offs & Alternatives Summary

| Decision | Option A | Option B | When to pick A | When to pick B |
|----------|----------|----------|----------------|----------------|
| Broker model | AMQP/JMS | Log-based | Per-message expensive work, no replay, fan-out task queue | High throughput, derived data, replay needed |
| Sync derived stores | Dual writes | CDC | Never — dual writes are broken under concurrency | Always, or use outbox pattern |
| State of truth | Mutable DB + CDC | Event sourcing | Existing app, minimal change | New app where intent matters, audit/replay critical |
| Time semantics | Processing time | Event time | Sub-second budget, delay rare | Correctness over simplicity |
| Stragglers | Drop | Publish correction | Approximate metrics | Billing, financial, audited outputs |
| Fault tolerance | Microbatching | Checkpoint barriers + idempotence | Spark ecosystem, ~1s latency OK | Low-latency, custom windows |
| Operator state | Remote KV | Local + replicated changelog | Tiny state, infra already exists | Large state, latency matters |

---

## Key Takeaways

1. **Brokers are databases optimized for streams.** The deepest design choice is whether reading is destructive (AMQP) or non-destructive (log). Replayability changes everything downstream.
2. **CDC turns the database's replication log into a public API.** Once you have a single ordered log of writes, you don't need dual writes — every other store becomes a follower.
3. **State is the integral of an event stream; a stream is the derivative of state.** This duality lets you derive many read views from one write log, including views you haven't designed yet.
4. **Time on streams has two coordinates** — event time and processing time — and confusing them creates phantom spikes during recovery. Watermarks and three-timestamp tricks tame the chaos but never eliminate it.
5. **Joins on streams come in three shapes** that differ in what state they keep and over what window. Stream–table joins via local CDC-fed hash tables avoid hammering the source DB.
6. **Exactly-once is really effectively-once** — achieved either by keeping the whole pipeline inside one transactional framework (Flink/Kafka/Dataflow) or by making external writes idempotent with monotonic offsets plus fencing.
7. **Replayability is the killer feature.** It lets you fix bugs, build new views, debug in production, and recover from operator crashes — all without redesigning the source of truth.

---

## See Also

- `[[01-message-brokers-amqp-vs-log]]` — broker design space
- `[[02-change-data-capture]]` — CDC pattern, dual-writes problem, outbox
- `[[03-event-sourcing-immutable-logs]]` — event sourcing, CQRS, crypto-shredding
- `[[04-stream-processing-use-cases]]` — CEP, analytics, materialized views, stream search, IVM
- `[[05-reasoning-about-time-and-windows]]` — event vs processing time, stragglers, window types
- `[[06-stream-joins]]` — three join shapes, time dependence, slowly changing dimensions
- `[[07-stream-fault-tolerance]]` — microbatching, checkpointing, atomic commit, idempotence